## Setting up your keys - OPTIONAL!

We're now going to try asking a bunch of models some questions!

This is totally optional. If you have keys to Anthropic, Gemini or others, then you can add them in.

If you'd rather not spend the extra, then just watch me do it!

For OpenAI, visit https://openai.com/api/  
For Anthropic, visit https://console.anthropic.com/  
For Google, visit https://aistudio.google.com/   
For DeepSeek, visit https://platform.deepseek.com/  
For Groq, visit https://console.groq.com/  
For Grok, visit https://console.x.ai/  


You can also use OpenRouter as your one-stop-shop for many of these! OpenRouter is "the unified interface for LLMs":

For OpenRouter, visit https://openrouter.ai/  


With each of the above, you typically have to navigate to:
1. Their billing page to add the minimum top-up (except Gemini, Groq, Google, OpenRouter may have free tiers)
2. Their API key page to collect your API key

# Day 1

In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

load_dotenv(override=True)
gemini_api_key = os.getenv("GOOGLE_API_KEY")
ollama_api_key=os.getenv("OLLAMA_API_KEY")
openai_api_key=os.getenv("OPENAI_API_KEY")

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

openai=OpenAI()
gemini = OpenAI(api_key=gemini_api_key, base_url = gemini_url)
ollama=OpenAI(api_key=ollama_api_key, base_url=ollama_url)



In [ ]:
# Test connection with a riddle
tell_a_joke = [
    {"role": "user", "content": "tell the funniest short joke you have about any topic"}
]

response= openai.chat.completions.create(model='gpt-4.1-mini', messages=tell_a_joke)
result = response.choices[0].message.content
display(Markdown(result))

response= gemini.chat.completions.create(model='gemini-2.5-flash', messages=tell_a_joke)
result = response.choices[0].message.content
display(Markdown(result))

response= ollama.chat.completions.create(model='deepseek-r1:1.5b', messages=tell_a_joke)
result = response.choices[0].message.content
display(Markdown(result))

## Routers and Abtraction Layers

Starting with the wonderful OpenRouter.ai - it can connect to all the models above!

Visit openrouter.ai and browse the models.

Here's one we haven't seen yet: GLM 4.5 from Chinese startup z.ai

## LLM Frameworks

Make sure to check Day 1 notebook to view prompt caching capability of liteLLM

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")
response = llm.invoke(tell_a_joke)
display(Markdown(response.content))

In [ ]:
from litellm import completion # compatible with bedrock or azure due to "provider/model" format
response = completion(model="openai/gpt-4.1-mini", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

In [ ]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

In [ ]:
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet=f.read()

loc=hamlet.find("Speak, man")
print(hamlet[loc:loc+50])


In [ ]:
question = [{"role": "user", "content": "In hamlet, when Laertes asks 'Where is my father', what is the reply?"}]

response = completion(model="gemini/gemini-2.5-flash", messages=question)
result = response.choices[0].message.content
display(Markdown(result)) # the response is confident but innaccurate

In [ ]:
# lets add context
question[0]['content'] += "\n\nFor context, here is the entire text of Hamlet:\n\n"+hamlet
response = completion(model="gemini/gemini-2.5-flash", messages=question)
result = response.choices[0].message.content
display(Markdown(result)) # returns the correct answer - if we were to rerun cached tokens would increase due to using very similar inputs

# Day 2 - Gradio Interfaces

In [ ]:
import gradio as gr

system_message = "You are a helpful assistant"

def message_gpt(prompt):
    messages =[
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    return response.choices[0].message.content






def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    stream = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages, stream=True)
    response=""
    display_handle= display(Markdown(""), display_id=True)

    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)



    

In [ ]:

message_gpt("What is todays date?")

In [ ]:
def shout(text):
    print(f"The function shout has been called with input: {text}")
    return text.upper()

shout("Hello, you mucnhkin")

In [ ]:
gr.Interface(fn=shout, inputs='textbox', outputs="textbox", flagging_mode='never').launch() # passing the function as a variable aka callback

# .launch(share=True) means huggungface will host your app but still use your machine for function calling (http tunnelling)

In [ ]:
# adding authentication
gr.Interface(fn=shout, inputs='textbox', outputs="textbox", flagging_mode='never').launch(inbrowser=True, auth=['alexander', 'bananas'])

In [ ]:
message_input = gr.Textbox(label="Your Message", info = "Enter a message to be shouted.", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=shout,
    title="Shout",
    inputs=[message_input],
    outputs=[message_output],
    examples = ['hello', 'howdy'],
    flagging_mode='never'
)

view.launch(auth=['alexander28', '!IBM2026'])


In [ ]:
# lets link in our message_gpt or stream_gpt

system_message = "You are a helpful assistant that responds in markdown without code blocks"

def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]

    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result=""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result






system_message = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your Message", info = "Enter your prompt for GPT-4.1-mini", lines=7)
message_output = gr.Markdown(label="GPT's Response")

view=gr.Interface(
    fn=stream_gpt,
    title="GPT",
    inputs = [message_input],
    outputs = [message_output],
    examples = ["Who is the current UK Prime Minister?", "Premier League Teams ranked by Titles"],
    flagging_mode='never'
)

view.launch()

## Multi-Model Gradio

In [ ]:
# create stream for gemini
def stream_gemini(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    stream = gemini.chat.completions.create(
        model='gemini-2.5-flash',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result+=chunk.choices[0].delta.content or ""
        yield result

def stream_model(prompt, model):
    if model == 'GPT':
        result = stream_gpt(prompt)
    elif model =='Gemini':
        result = stream_gemini(prompt)
    else:
        raise ValueError("Unknown Model!")
    yield from result

In [ ]:
message_input = gr.Textbox(label="Your Message", info = "Send a message to a model of your choice", lines=7)
model_selector = gr.Dropdown(["GPT", "Gemini"], label="Select Model", value='GPT')
message_output = gr.Textbox(label='Response:')

view = gr.Interface(
    fn=stream_model,
    title="LLMs",
    inputs = [message_input, model_selector],
    outputs = [message_output],
    examples = [["Who is the current UK Prime Minister?", "GPT"], ["Premier League Teams ranked by Titles", 'Gemini']],
    flagging_mode='never'

)

view.launch()

## Creating a brochure

In [ ]:
from scraper import fetch_website_contents

In [ ]:

# Again this is typical Experimental mindset - I'm changing the global variable we used above:

system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Gemini":
        result = stream_gemini(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
company_input = gr.Textbox(label="Enter Company name")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT", "Gemini"], label="Select Model", value='GPT')
message_output = gr.Markdown(label='Response:')

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator",
    inputs = [company_input, url_input, model_selector],
    outputs = [message_output],
    examples = [
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Edward Donner", "https://edwarddonner.com", "Claude"]
        ],
    flagging_mode='never'

)

view.launch()

## Chat UIs

Objectives:
* On Brand Persona
* Maintain context between messages (using System Prompt, Context Insertion, Multi-shot Prompting)
* Subject Matter Expertise

In [ ]:
MODEL = "gpt-4.1-mini"
system_message = "You are a helpful assistant"

In [ ]:
# lets define a callback function called chat(message, history) -> messages from the existing ui will be passed in as history
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content":system_message}] + history + [{"role":"user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delte.content or ""
        yield result

gr.ChatInterface(fn=chat,type="messages").launch()

In [ ]:
# one shot prompt 
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

# context injection 
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

# Day 4

## Tool Calling in Theory

- An LLM decides and executes code on my computer

## Tool Calling in Practice

- Code invokes LLM with context and question, LLM informs code for a tool call. Code executes the tool and return chat history, context, tool call output and the question. LLM responds with the answer

Common Use Cases for Tools:

* Fetching Data
* Take Action like bookings
* Perform calculations or run code (Coding agents can write AND execute code)
* Modify the UI to interact with Users
* In agentic AI, LLMs can use 


In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

load_dotenv(override=True)
gemini_api_key = os.getenv("GOOGLE_API_KEY")
ollama_api_key=os.getenv("OLLAMA_API_KEY")
openai_api_key=os.getenv("OPENAI_API_KEY")

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

openai=OpenAI()
gemini = OpenAI(api_key=gemini_api_key, base_url = gemini_url)
ollama=OpenAI(api_key=ollama_api_key, base_url=ollama_url)

OPENAI_MODEL = 'gpt-4.1-mini'

In [ ]:
system_message = """ 
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 senetence unless necessary.
Always be accurate, if you do not know the answer then say so.
Prices are fixed regardless of dates so you do not need to ask for them
"""

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "systems", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages)
    return response.choices[0].message.content


### Building tools

In [ ]:
ticket_prices = {"london": "£800", "new york": "£1200", "amsterdam": "£250", "berlin": "£400"}

# for loops with a dict automatically iterates over keys
def get_ticket_price(destination):
    print(f"Tool called for {destination}")
    price = ticket_prices.get(destination.lower(), "Unknown City") # .get() looks up key in dictionary and finds value
    return f"The price of a ticket to {destination} is {price}"

# get_ticket_price("london")

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The city that the customer wants to travel to,"
            },
        },
        "required": ['destination'],
        "additionalProperties": False
    }
}

# create tool list 
tools = [{"type": "function", "function": price_function}]

Conceptual chat completion response  looks like 
{
  "object": "chat.completion",

  "id": "chatcmpl-abc123",
  
  "model": "gpt-6-astra",
  
  "created": 1738960610,
  
  "request_id": "req_ded8ab984ec4bf840f37566c1011c417",
  
  "tool_choice": null,
  
  "usage": {
  
    "total_tokens": 31,
  
    "completion_tokens": 18,
  
    "prompt_tokens": 13
  
  },
  
  "seed": 4944116822809979520,
  
  "top_p": 1.0,
  
  "temperature": 1.0,
  
  "presence_penalty": 0.0,
  
  "frequency_penalty": 0.0,
  
  "system_fingerprint": "fp_50cad350e4",
  
  "input_user": null,
  
  "service_tier": "default",
  
  "tools": null,
  
  "metadata": {},
  
  "choices": [
  
    {
  
      "index": 0,
  
      "message": {
  
        "content": "Mind of circuits hum,  \nLearning patterns in silence—  \nFuture's quiet spark.",
  
        "role": "assistant",
  
        "tool_calls": null,
  
        "function_call": null
  
      },
  
      "finish_reason": "stop",
  
      "logprobs": null

In [ ]:
# write function to execute tool call
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get("destination")
        price_details=get_ticket_price(city)
        response = {
            "role":"tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response


# new chat function 
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages, tools=tools)
    
    if response.choices[0].finish_reason =="tool_calls": # if llm response is a tool call we will execute the tool call and store output in response. Append tool call message and output to our messages and invoke LLM again
        message = response.choices[0].message #
        response=handle_tool_call(message) # execute code and store result
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages) # no more tool call
    
    return response.choices[0].message.content


In [ ]:
import gradio as gr

gr.ChatInterface(fn=chat, type='messages').launch()

## Handling multiple tool calls

In [ ]:
system_message = """ 
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 senetence unless necessary.
Always be accurate, if you do not know the answer then say so.
Prices are fixed regardless of dates so you do not need to ask for them
"""

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination')
            price = get_ticket_price(city)
            responses .append({
            "role":"tool",
            "content": price,
            "tool_call_id": tool_call.id
        })
    return responses

def chat(message, history):
    print("HISTORY:", repr(history))
    print("TYPE:", type(history))
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages, tools = tools)

    if response.choices[0].finish_reason =='tool_calls':
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages)

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type='messages').launch()

In [ ]:
# incorporate sqlite
import sqlite3
DB = "prices.db"
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices(city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: getting price ticket for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor=conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE city = ?", (city.lower(),)) # ? is a parameter holder fo values we will provide
        result = cursor.fetchone()

        return f"Ticket price for {city} is ${result[0]}" if result else "No price data available for this city"

get_ticket_price('london')


In [ ]:
ticket_prices = {'london': 799, 'berlin':600, "new york": 1200, "amsterdam": 300}

def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT (city) DO UPDATE SET price = ?", (city.lower(), price, price))
        conn.commit()

for city, price in ticket_prices.items():
    set_ticket_price(city, price)

get_ticket_price('london')


In [ ]:
# as we have update functions with same name no code change is required
gr.ChatInterface(fn=chat, type='messages').launch()

## Exercise: Set & Get Function

In [53]:
set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The city that the admin wants to set the price for"
            },
            "price": {
                "type": "number",
                "description": "The price of the return ticket"
            }
        },
        "required": ["destination", "price"],
        "additionalProperties": False
    }
}

get_price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The city that the customer wants to travel to,"
            },
        },
        "required": ['destination'],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": get_price_function}, {"type": "function", "function": set_price_function}]

In [63]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting ticket price for {city}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE city = ?", (city.lower(),))
        result = cursor.fetchone()
    
    return f"Ticket price for {city} is ${result[0]}" if result else "No data available for this city. Admin must insert data."

def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: Setting new price for {city} as ${price}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [64]:
available_functions = {
    "get_ticket_price": get_ticket_price,
    "set_ticket_price": set_ticket_price

}

def handle_tool_calls(message):
    print(message)
    responses = []
    
    for tc in message.tool_calls:
        fn_name = tc.function.name
        fn_to_call = available_functions.get(fn_name)
        print(fn_to_call)
        
        if not fn_to_call:
            responses.append({
                "role":"tool",
                "content": f"Error: Function {fn_name} not found",
                "tool_call_id": tc.id
            })
            continue

        arguments = json.loads(tc.function.arguments)

        if fn_name == "get_ticket_price":
            city = arguments.get('destination')
            price = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price,
                "tool_call_id": tc.id
            })

        elif fn_name == "set_ticket_price":
            city = arguments.get('destination')
            price = arguments.get('price')
            fn_to_call(city, price)
            responses.append({
                "role": "tool",
                "content": f"{city} ticket price updated to ${price}",
                "tool_call_id": tc.id
            })
    return responses


    def chat(message, history):
        history =[{"role": h["role"], "content": h["content"]} for h in history]
        messages = [{"role": "system", "content": system_message}] + history + [{"role":"user", "content": message}]
        response = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages, tools=tools)

        while response.choices[0].finish_reason == "tool_calls":
            messages = response.choices[0].message
            responses = handle_tool_calls(message)
            messages.append(message)
            messages.extend(responses)
            response = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages)

        return response.choices[0].message.content






In [ ]:
gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


HISTORY: []
TYPE: <class 'list'>
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_jsyoFNqBJ8u7KSiBbUG4DAKJ', function=Function(arguments='{"destination":"oslo"}', name='get_ticket_price'), type='function')])
<function get_ticket_price at 0x144f66660>
DATABASE TOOL CALLED: Getting ticket price for oslo
HISTORY: [{'role': 'user', 'metadata': None, 'content': 'Hello what is the ticket price for oslo', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': "I'm sorry, but we currently do not have ticket price information for Oslo.", 'options': None}]
TYPE: <class 'list'>
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_1DQTNreGg7j5z315UjsbLT5Y', function=Function(arguments='{"destination":"Oslo","price":300}', name='set_ticket_p

# DAY 5

Agent - An LLM agent that can run loops of LLM and Tool Calls based on environment feedback to complete a task

* Covers Text-to-speech
* Image Generation
* Gradio Blocks